1. R 通道的本质是星座图（Constellation Diagram），但对其进行了统计密度化处理。

2. 信号在复平面（Complex Plane）上的空间分布。复平面由实部（In-phase, I）作为 X 轴，虚部（Quadrature, Q）作为 Y 轴

3. 统计密度化处理：区别于一般星座图，这里是绘制为二维直方图，颜色亮度越高，代表此处出现的次数越多

假设我们接收到的离散数字信号序列为 $S$，长度为 $N$：
$$ S = \{s[0], s[1], \dots, s[N-1]\} $$
其中每一个采样点 $s[n]$ 都是复数：
$$ s[n] = I[n] + jQ[n] $$

我们需要构建一个 $H \times W$ （例如 $64 \times 64$）的图像矩阵 $\mathbf{R}$。

**步骤 1：归一化与离散化 (Quantization)**
我们需要将连续的 $I, Q$ 数值映射到图像的整数坐标 $(u, v)$ 上。设定观察范围为 $[-B, B]$（例如 $[-1.2, 1.2]$）。

$$ u[n] = \lfloor \frac{I[n] - (-B)}{2B} \times (W-1) \rfloor $$
$$ v[n] = \lfloor \frac{Q[n] - (-B)}{2B} \times (H-1) \rfloor $$

**步骤 2：统计累加 (Histogramming)**
图像矩阵 $\mathbf{R}$ 中的每一个像素值 $R_{u,v}$ 等于落入该坐标点的采样点个数：
$$ R_{u,v} = \sum_{n=0}^{N-1} \delta(u[n] - u, v[n] - v) $$
其中 $\delta$ 是克罗内克函数（相等为1，不等为0）。

**步骤 3：动态范围压缩 (Log-Compression)**
由于信号中心点的密度可能极高（比如 5000 次），而边缘轨迹可能很低（比如 5 次），直接归一化会导致边缘不可见。因此我们取对数：
$$ \mathbf{R}'_{u,v} = \log(1 + R_{u,v}) $$

**步骤 4：最终归一化**
将数值映射到 $[0, 1]$ 区间供神经网络读取：
$$ \mathbf{R}_{final} = \frac{\mathbf{R}' - \min(\mathbf{R}')}{\max(\mathbf{R}') - \min(\mathbf{R}')} $$


BPSK：图像水平中轴线上有两个高亮的光斑（左右各一个）。如果信噪比低，是两团模糊的云；如果信噪比高，是两个锐利的点。
QPSK：图像上有四个光斑，呈正方形排列。
8PSK：图像上有八个光斑，围成一个圆圈。
16QAM：图像上有 4x4 = 16 个光斑，呈网格状排列。
FSK：这是一个特例。因为 FSK 是恒定幅度、相位连续旋转的，所以在复平面上，它画出的是一个圆环（Ring）。
噪声的影响：
低信噪比 (-5dB)：光斑会扩散、粘连在一起，背景全是灰蒙蒙的噪点。
高信噪比 (25dB)：光斑收缩成针尖大小，背景全黑（纯净

In [9]:
import numpy as np
import torch
import torch.nn.functional as F
import os

# ================= 关键修改：静默后端 =================
import matplotlib
# 设置为 'Agg' 后端，确保不会弹出显示窗口，只进行文件生成
matplotlib.use('Agg') 
import matplotlib.pyplot as plt

# ================= 配置 =================
CLASS_NAMES = ["BPSK", "QPSK", "PSK8", "QAM16", "QAM64", "FSK2"]
# SNR_LEVELS = [-5, 2, 10, 18, 25]
SNR_LEVELS = [-5, -3, -1, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41]   
IMG_SIZE = 512     # <--- 修改：高分辨率 512x512
SAMPLES = 10240    # 保持长曝光点数

# 创建保存目录
SAVE_DIR = "./r_channel_512_demo"
os.makedirs(SAVE_DIR, exist_ok=True)

# ================= 信号生成器 (保持不变) =================
def generate_rrc_filter(N_taps, beta, sps):
    t = torch.arange(-(N_taps - 1) // 2, (N_taps - 1) // 2 + 1, dtype=torch.float32) / sps
    mid_idx = (N_taps - 1) // 2
    t[mid_idx] = 1e-8
    num = torch.sin(np.pi * t * (1 - beta)) + 4 * beta * t * torch.cos(np.pi * t * (1 + beta))
    den = np.pi * t * (1 - (4 * beta * t)**2)
    coeffs = num / den
    coeffs[mid_idx] = 1 - beta + (4 * beta / np.pi)
    return coeffs / torch.sqrt(torch.sum(coeffs**2))

def add_noise_and_imperfect(sig, snr_db):
    N = len(sig)
    freq_offset = np.random.uniform(-50, 50) 
    t = torch.arange(N, dtype=torch.float32) / 80e3
    phase_offset = np.random.uniform(0, 2*np.pi)
    rotation = torch.exp(1j * (2 * np.pi * freq_offset * t + phase_offset))
    sig = sig * rotation
    
    sig_power = torch.mean(torch.abs(sig)**2)
    noise_power = sig_power / (10**(snr_db/10.0))
    noise = (torch.randn_like(sig) + 1j*torch.randn_like(sig)) * torch.sqrt(noise_power/2)
    return sig + noise

def generate_signal_for_demo(mod_type, snr_db):
    N = SAMPLES
    Fs, Rs = 80e3, 20e3
    sps = int(Fs / Rs)
    num_symbols = int(N / sps) + 50
    
    if mod_type == "BPSK":
        syms = 2*torch.randint(0, 2, (num_symbols,)) - 1 + 0j
    elif mod_type == "QPSK":
        phase = np.pi/4 + torch.randint(0, 4, (num_symbols,)) * np.pi/2
        syms = torch.exp(1j * phase)
    elif mod_type == "PSK8":
        phase = torch.randint(0, 8, (num_symbols,)) * 2*np.pi/8
        syms = torch.exp(1j * phase)
    elif mod_type == "QAM16":
        idx = torch.randint(0, 16, (num_symbols,))
        syms = ((idx // 4) * 2 - 3) + 1j*((idx % 4) * 2 - 3)
    elif mod_type == "QAM64":
        idx = torch.randint(0, 64, (num_symbols,))
        syms = ((idx // 8) * 2 - 7) + 1j*((idx % 8) * 2 - 7)
    elif mod_type == "FSK2":
        bits = torch.randint(0, 2, (N + 100,))
        f_dev = np.random.uniform(500, 6000)
        freqs = (bits * 2 - 1) * f_dev
        phase = 2 * np.pi * torch.cumsum(freqs/Fs, dim=0)
        sig = torch.exp(1j * phase)[:N]
        return add_noise_and_imperfect(sig, snr_db)

    upsampled = torch.zeros(num_symbols * sps, dtype=torch.complex64)
    upsampled[::sps] = syms
    taps = 10 * sps + 1
    h = generate_rrc_filter(taps, 0.35, sps)
    real = F.conv1d(upsampled.real.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    imag = F.conv1d(upsampled.imag.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    sig = (real + 1j*imag).squeeze()
    if len(sig) > N: sig = sig[:N]
    
    return add_noise_and_imperfect(sig, snr_db)

# ================= R 通道计算核心 =================
def compute_r_channel(sig):
    """
    计算 R 通道：星座图密度
    Output: 2D Tensor (512, 512)
    """
    # 1. 归一化 
    scale = torch.sqrt(torch.mean(sig.real**2 + sig.imag**2)) + 1e-9
    norm_sig = sig / (3.0 * scale)
    
    x = norm_sig.real
    y = norm_sig.imag
    
    range_min, range_max = -1.2, 1.2
    x = torch.clamp(x, range_min, range_max)
    y = torch.clamp(y, range_min, range_max)
    
    # 2. 映射到 grid (512x512)
    bins = IMG_SIZE
    x_idx = ((x - range_min) / (range_max - range_min) * (bins - 1)).long()
    y_idx = ((y - range_min) / (range_max - range_min) * (bins - 1)).long()
    
    # 3. 统计频次
    flat_idx = y_idx * bins + x_idx
    hist = torch.bincount(flat_idx, minlength=bins*bins).float()
    hist = hist.view(bins, bins)
    
    # 4. 对数压缩
    hist = torch.log1p(hist)
    
    # 5. 归一化 [0, 1]
    hist = (hist - hist.min()) / (hist.max() - hist.min() + 1e-8)
    
    # 6. 翻转 Y 轴
    return torch.flip(hist, [0])

# ================= 主流程 =================
def main():
    print(f"开始生成 R 通道演示图 (分辨率: {IMG_SIZE}x{IMG_SIZE})...")
    print(f"保存路径: {os.path.abspath(SAVE_DIR)}")
    
    # 准备一张大图用于总览 (虽然不显示，但可以存下来方便看)
    fig, axes = plt.subplots(len(CLASS_NAMES), len(SNR_LEVELS), figsize=(20, 24))
    
    for r, mod in enumerate(CLASS_NAMES):
        for c, snr in enumerate(SNR_LEVELS):
            # 1. 生成信号
            sig = generate_signal_for_demo(mod, snr)
            
            # 2. 计算 R 通道 (512x512)
            heatmap = compute_r_channel(sig)
            
            # 3. 保存单张图片 (严格像素对齐)
            # 使用 plt.imsave 可以直接保存数组为图片，不带坐标轴，像素尺寸严格匹配
            filename = f"{SAVE_DIR}/{mod}_SNR{snr}.png"
            plt.imsave(filename, heatmap.numpy(), cmap='gray')
            print(f"已保存: {filename}")
            
            # 4. 绘制到总览图 (仅用于 check)
            ax = axes[r, c]
            ax.imshow(heatmap.numpy(), cmap='inferno')
            if r == 0: ax.set_title(f"SNR = {snr} dB", fontsize=16)
            if c == 0: ax.set_ylabel(mod, fontsize=20, fontweight='bold')
            ax.set_xticks([])
            ax.set_yticks([])
            
    # 保存总览图
    overview_path = f"{SAVE_DIR}/R_Channel_Overview_512.png"
    plt.savefig(overview_path, dpi=100, bbox_inches='tight')
    plt.close(fig) # 显式关闭，释放内存
    
    print(f"全部完成！")
    print(f"单张高清图 (512x512) 在: {SAVE_DIR}")
    print(f"总览图在: {overview_path}")

if __name__ == "__main__":
    main()

开始生成 R 通道演示图 (分辨率: 512x512)...
保存路径: e:\Project\AMR-Benchmark\Other\r_channel_512_demo
已保存: ./r_channel_512_demo/BPSK_SNR-5.png
已保存: ./r_channel_512_demo/BPSK_SNR-3.png
已保存: ./r_channel_512_demo/BPSK_SNR-1.png
已保存: ./r_channel_512_demo/BPSK_SNR1.png
已保存: ./r_channel_512_demo/BPSK_SNR3.png
已保存: ./r_channel_512_demo/BPSK_SNR5.png
已保存: ./r_channel_512_demo/BPSK_SNR7.png
已保存: ./r_channel_512_demo/BPSK_SNR9.png
已保存: ./r_channel_512_demo/BPSK_SNR11.png
已保存: ./r_channel_512_demo/BPSK_SNR13.png
已保存: ./r_channel_512_demo/BPSK_SNR15.png
已保存: ./r_channel_512_demo/BPSK_SNR17.png
已保存: ./r_channel_512_demo/BPSK_SNR19.png
已保存: ./r_channel_512_demo/BPSK_SNR21.png
已保存: ./r_channel_512_demo/BPSK_SNR23.png
已保存: ./r_channel_512_demo/BPSK_SNR25.png
已保存: ./r_channel_512_demo/BPSK_SNR27.png
已保存: ./r_channel_512_demo/BPSK_SNR29.png
已保存: ./r_channel_512_demo/BPSK_SNR31.png
已保存: ./r_channel_512_demo/BPSK_SNR33.png
已保存: ./r_channel_512_demo/BPSK_SNR35.png
已保存: ./r_channel_512_demo/BPSK_SNR37.png
已保存: ./r_channel

C:\Users\CPC-ML\AppData\Local\Temp\ipykernel_13956\767052946.py:63: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  syms = ((idx // 4) * 2 - 3) + 1j*((idx % 4) * 2 - 3)


已保存: ./r_channel_512_demo/QAM16_SNR-5.png
已保存: ./r_channel_512_demo/QAM16_SNR-3.png
已保存: ./r_channel_512_demo/QAM16_SNR-1.png
已保存: ./r_channel_512_demo/QAM16_SNR1.png
已保存: ./r_channel_512_demo/QAM16_SNR3.png
已保存: ./r_channel_512_demo/QAM16_SNR5.png
已保存: ./r_channel_512_demo/QAM16_SNR7.png
已保存: ./r_channel_512_demo/QAM16_SNR9.png
已保存: ./r_channel_512_demo/QAM16_SNR11.png
已保存: ./r_channel_512_demo/QAM16_SNR13.png
已保存: ./r_channel_512_demo/QAM16_SNR15.png
已保存: ./r_channel_512_demo/QAM16_SNR17.png
已保存: ./r_channel_512_demo/QAM16_SNR19.png
已保存: ./r_channel_512_demo/QAM16_SNR21.png
已保存: ./r_channel_512_demo/QAM16_SNR23.png
已保存: ./r_channel_512_demo/QAM16_SNR25.png
已保存: ./r_channel_512_demo/QAM16_SNR27.png
已保存: ./r_channel_512_demo/QAM16_SNR29.png
已保存: ./r_channel_512_demo/QAM16_SNR31.png
已保存: ./r_channel_512_demo/QAM16_SNR33.png
已保存: ./r_channel_512_demo/QAM16_SNR35.png
已保存: ./r_channel_512_demo/QAM16_SNR37.png
已保存: ./r_channel_512_demo/QAM16_SNR39.png
已保存: ./r_channel_512_demo/QAM16_SNR41.p

C:\Users\CPC-ML\AppData\Local\Temp\ipykernel_13956\767052946.py:66: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  syms = ((idx // 8) * 2 - 7) + 1j*((idx % 8) * 2 - 7)


已保存: ./r_channel_512_demo/QAM64_SNR1.png
已保存: ./r_channel_512_demo/QAM64_SNR3.png
已保存: ./r_channel_512_demo/QAM64_SNR5.png
已保存: ./r_channel_512_demo/QAM64_SNR7.png
已保存: ./r_channel_512_demo/QAM64_SNR9.png
已保存: ./r_channel_512_demo/QAM64_SNR11.png
已保存: ./r_channel_512_demo/QAM64_SNR13.png
已保存: ./r_channel_512_demo/QAM64_SNR15.png
已保存: ./r_channel_512_demo/QAM64_SNR17.png
已保存: ./r_channel_512_demo/QAM64_SNR19.png
已保存: ./r_channel_512_demo/QAM64_SNR21.png
已保存: ./r_channel_512_demo/QAM64_SNR23.png
已保存: ./r_channel_512_demo/QAM64_SNR25.png
已保存: ./r_channel_512_demo/QAM64_SNR27.png
已保存: ./r_channel_512_demo/QAM64_SNR29.png
已保存: ./r_channel_512_demo/QAM64_SNR31.png
已保存: ./r_channel_512_demo/QAM64_SNR33.png
已保存: ./r_channel_512_demo/QAM64_SNR35.png
已保存: ./r_channel_512_demo/QAM64_SNR37.png
已保存: ./r_channel_512_demo/QAM64_SNR39.png
已保存: ./r_channel_512_demo/QAM64_SNR41.png
已保存: ./r_channel_512_demo/FSK2_SNR-5.png
已保存: ./r_channel_512_demo/FSK2_SNR-3.png
已保存: ./r_channel_512_demo/FSK2_SNR-1.png


1. G 通道 的核心原理源自非线性动力学中的**“延迟嵌入” (Delay Embedding)** 或 “相空间重构” (Phase Space Reconstruction)。（实部动力学轨迹）

*   **物理定义**：它绘制的是信号实部在当前时刻的值 $I[t]$ 与前一时刻的值 $I[t-1]$ 之间的关系。
    *   X 轴：$I[t]$ (Current)
    *   Y 轴：$I[t-1]$ (Previous)
*   **物理意义**：这描述了信号的**“变化率”**或**“惯性”**。
    *   **BPSK/QPSK**：信号主要在几个固定电平之间跳变。经过 RRC 滤波后，这种跳变是平滑的直线或曲线。
    *   **FSK**：信号的实部 $I(t) = \cos(\omega t + \phi)$。一个余弦波在 $(x(t), x(t-1))$ 坐标系下，画出来是一个**椭圆**（李萨如图形）。
    *   **噪声**：噪声是随机跳变的，它会填充整个空间，形成均匀的雾霾。
*   **为什么要这个通道？**：这是区分 **PSK（数字调制，有突变）** 和 **FSK（模拟调频，连续变化）** 的最强特征。即使星座图因为信道旋转了，这种“突变 vs 渐变”的动力学特征依然存在。

输入信号序列为复数 $s[n] = I[n] + jQ[n]$。对于 G 通道，我们只关注实部 $I[n]$。

**步骤 1：构造延迟向量**
我们需要构建点对 $(x, y)$：
$$ x[n] = I[n] $$
$$ y[n] = I[n-1] $$
(代码中为了向量化计算，通常取 `real[1:]` 和 `real[:-1]`)

**步骤 2：离散化与映射**
设定观察范围 $[-B, B]$，映射到 $512 \times 512$ 的网格：
$$ u[n] = \text{Map}(I[n]) $$
$$ v[n] = \text{Map}(I[n-1]) $$

**步骤 3：密度统计 (2D Histogram)**
$$ \mathbf{G}_{u,v} = \sum_{n=1}^{N-1} \delta(u[n] - u, v[n] - v) $$

**步骤 4：对数增强**
$$ \mathbf{G}_{final} = \text{Normalize}(\log(1 + \mathbf{G})) $$


*   **BPSK**：
    *   如果信号保持不变（$1 \to 1$ 或 $-1 \to -1$），会在右上角 $(1,1)$ 和左下角 $(-1,-1)$ 形成**高亮斑点**。
    *   如果信号跳变（$1 \to -1$ 或 $-1 \to 1$），会在两个斑点之间画出**连接线**。
    *   **形状**：像一个哑铃，或者是对角线上的两个点中间连着线。
*   **QAM / QPSK**：
    *   由于电平级数多了（比如 -3, -1, 1, 3），会有多个斑点分布在对角线附近。
    *   斑点之间有复杂的网格状连接线。
*   **FSK (关键特征)**：
    *   由于 $I(t)$ 是正弦波，$(I_t, I_{t-1})$ 会画出一个完美的**空心椭圆**或**圆圈**。
    *   **区别**：PSK 是实心的、有对角线趋势的；FSK 是空心的、环状的。这是 CNN 识别 FSK 的核心依据。
*   **噪声**：
    *   噪声会在主对角线附近形成弥散的云团（因为 $I_t$ 和 $I_{t-1}$ 有相关性）。


In [10]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg') # 静默模式
import matplotlib.pyplot as plt
import os

# ================= 配置 =================
CLASS_NAMES = ["BPSK", "QPSK", "PSK8", "QAM16", "QAM64", "FSK2"]
# SNR_LEVELS = [-5, 2, 10, 18, 25]
SNR_LEVELS = [-5, -3, -1, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41]   
IMG_SIZE = 512
SAMPLES = 10240

SAVE_DIR = "./g_channel_demo"
os.makedirs(SAVE_DIR, exist_ok=True)

# ================= 信号生成 (复用之前的高保真逻辑) =================
def generate_rrc_filter(N_taps, beta, sps):
    t = torch.arange(-(N_taps - 1) // 2, (N_taps - 1) // 2 + 1, dtype=torch.float32) / sps
    mid_idx = (N_taps - 1) // 2
    t[mid_idx] = 1e-8
    num = torch.sin(np.pi * t * (1 - beta)) + 4 * beta * t * torch.cos(np.pi * t * (1 + beta))
    den = np.pi * t * (1 - (4 * beta * t)**2)
    coeffs = num / den
    coeffs[mid_idx] = 1 - beta + (4 * beta / np.pi)
    return coeffs / torch.sqrt(torch.sum(coeffs**2))

def add_noise_and_imperfect(sig, snr_db):
    N = len(sig)
    # 极慢速频偏 (保持特征稳定)
    freq_offset = np.random.uniform(-50, 50) 
    t = torch.arange(N, dtype=torch.float32) / 80e3
    phase_offset = np.random.uniform(0, 2*np.pi)
    rotation = torch.exp(1j * (2 * np.pi * freq_offset * t + phase_offset))
    sig = sig * rotation
    
    sig_power = torch.mean(torch.abs(sig)**2)
    noise_power = sig_power / (10**(snr_db/10.0))
    noise = (torch.randn_like(sig) + 1j*torch.randn_like(sig)) * torch.sqrt(noise_power/2)
    return sig + noise

def generate_signal_for_demo(mod_type, snr_db):
    N = SAMPLES
    Fs, Rs = 80e3, 20e3
    sps = int(Fs / Rs)
    num_symbols = int(N / sps) + 50
    
    if mod_type == "BPSK":
        syms = 2*torch.randint(0, 2, (num_symbols,)) - 1 + 0j
    elif mod_type == "QPSK":
        phase = np.pi/4 + torch.randint(0, 4, (num_symbols,)) * np.pi/2
        syms = torch.exp(1j * phase)
    elif mod_type == "PSK8":
        phase = torch.randint(0, 8, (num_symbols,)) * 2*np.pi/8
        syms = torch.exp(1j * phase)
    elif mod_type == "QAM16":
        idx = torch.randint(0, 16, (num_symbols,))
        syms = ((idx // 4) * 2 - 3) + 1j*((idx % 4) * 2 - 3)
    elif mod_type == "QAM64":
        idx = torch.randint(0, 64, (num_symbols,))
        syms = ((idx // 8) * 2 - 7) + 1j*((idx % 8) * 2 - 7)
    elif mod_type == "FSK2":
        bits = torch.randint(0, 2, (N + 100,))
        f_dev = np.random.uniform(500, 6000)
        freqs = (bits * 2 - 1) * f_dev
        phase = 2 * np.pi * torch.cumsum(freqs/Fs, dim=0)
        sig = torch.exp(1j * phase)[:N]
        return add_noise_and_imperfect(sig, snr_db)

    upsampled = torch.zeros(num_symbols * sps, dtype=torch.complex64)
    upsampled[::sps] = syms
    taps = 10 * sps + 1
    h = generate_rrc_filter(taps, 0.35, sps)
    real = F.conv1d(upsampled.real.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    imag = F.conv1d(upsampled.imag.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    sig = (real + 1j*imag).squeeze()
    if len(sig) > N: sig = sig[:N]
    
    return add_noise_and_imperfect(sig, snr_db)

# ================= G 通道计算核心 (Real Trajectory) =================
def compute_g_channel(sig):
    """
    计算 G 通道：实部延迟嵌入 (I[t] vs I[t-1])
    Input: Complex Tensor (N,)
    Output: 2D Tensor (512, 512)
    """
    # 1. 归一化
    scale = torch.sqrt(torch.mean(sig.real**2 + sig.imag**2)) + 1e-9
    norm_sig = sig / (3.0 * scale)
    
    # 只取实部
    real = norm_sig.real
    
    # 2. 构造延迟对 (Current vs Previous)
    # x轴: I[t], y轴: I[t-1]
    x = real[1:]
    y = real[:-1]
    
    range_min, range_max = -1.2, 1.2
    x = torch.clamp(x, range_min, range_max)
    y = torch.clamp(y, range_min, range_max)
    
    # 3. 映射到 grid
    bins = IMG_SIZE
    x_idx = ((x - range_min) / (range_max - range_min) * (bins - 1)).long()
    y_idx = ((y - range_min) / (range_max - range_min) * (bins - 1)).long()
    
    # 4. 统计频次
    flat_idx = y_idx * bins + x_idx
    hist = torch.bincount(flat_idx, minlength=bins*bins).float()
    hist = hist.view(bins, bins)
    
    # 5. 对数压缩 + 归一化
    hist = torch.log1p(hist)
    hist = (hist - hist.min()) / (hist.max() - hist.min() + 1e-8)
    
    return torch.flip(hist, [0])

# ================= 主流程 =================
def main():
    print(f"开始生成 G 通道 (实部相空间) 演示图...")
    print(f"分辨率: {IMG_SIZE}x{IMG_SIZE}, 路径: {os.path.abspath(SAVE_DIR)}")
    
    fig, axes = plt.subplots(len(CLASS_NAMES), len(SNR_LEVELS), figsize=(20, 24))
    
    for r, mod in enumerate(CLASS_NAMES):
        for c, snr in enumerate(SNR_LEVELS):
            sig = generate_signal_for_demo(mod, snr)
            
            # 计算 G 通道
            heatmap = compute_g_channel(sig)
            
            # 保存单张
            filename = f"{SAVE_DIR}/{mod}_SNR{snr}.png"
            plt.imsave(filename, heatmap.numpy(), cmap='gray')
            print(f"已保存: {filename}")
            
            # 总览
            ax = axes[r, c]
            ax.imshow(heatmap.numpy(), cmap='inferno')
            if r == 0: ax.set_title(f"SNR = {snr} dB", fontsize=16)
            if c == 0: ax.set_ylabel(mod, fontsize=20, fontweight='bold')
            ax.set_xticks([])
            ax.set_yticks([])
            
    # 保存总览
    plt.savefig(f"{SAVE_DIR}/G_Channel_Overview.png", dpi=100, bbox_inches='tight')
    plt.close(fig)
    print("全部完成！")

if __name__ == "__main__":
    main()

开始生成 G 通道 (实部相空间) 演示图...
分辨率: 512x512, 路径: e:\Project\AMR-Benchmark\Other\g_channel_demo
已保存: ./g_channel_demo/BPSK_SNR-5.png
已保存: ./g_channel_demo/BPSK_SNR-3.png
已保存: ./g_channel_demo/BPSK_SNR-1.png
已保存: ./g_channel_demo/BPSK_SNR1.png
已保存: ./g_channel_demo/BPSK_SNR3.png
已保存: ./g_channel_demo/BPSK_SNR5.png
已保存: ./g_channel_demo/BPSK_SNR7.png
已保存: ./g_channel_demo/BPSK_SNR9.png
已保存: ./g_channel_demo/BPSK_SNR11.png
已保存: ./g_channel_demo/BPSK_SNR13.png
已保存: ./g_channel_demo/BPSK_SNR15.png
已保存: ./g_channel_demo/BPSK_SNR17.png
已保存: ./g_channel_demo/BPSK_SNR19.png
已保存: ./g_channel_demo/BPSK_SNR21.png
已保存: ./g_channel_demo/BPSK_SNR23.png
已保存: ./g_channel_demo/BPSK_SNR25.png
已保存: ./g_channel_demo/BPSK_SNR27.png
已保存: ./g_channel_demo/BPSK_SNR29.png
已保存: ./g_channel_demo/BPSK_SNR31.png
已保存: ./g_channel_demo/BPSK_SNR33.png
已保存: ./g_channel_demo/BPSK_SNR35.png
已保存: ./g_channel_demo/BPSK_SNR37.png
已保存: ./g_channel_demo/BPSK_SNR39.png
已保存: ./g_channel_demo/BPSK_SNR41.png
已保存: ./g_channel_demo/QPSK_S

C:\Users\CPC-ML\AppData\Local\Temp\ipykernel_13956\1307707760.py:60: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  syms = ((idx // 4) * 2 - 3) + 1j*((idx % 4) * 2 - 3)


已保存: ./g_channel_demo/QAM16_SNR-1.png
已保存: ./g_channel_demo/QAM16_SNR1.png
已保存: ./g_channel_demo/QAM16_SNR3.png
已保存: ./g_channel_demo/QAM16_SNR5.png
已保存: ./g_channel_demo/QAM16_SNR7.png
已保存: ./g_channel_demo/QAM16_SNR9.png
已保存: ./g_channel_demo/QAM16_SNR11.png
已保存: ./g_channel_demo/QAM16_SNR13.png
已保存: ./g_channel_demo/QAM16_SNR15.png
已保存: ./g_channel_demo/QAM16_SNR17.png
已保存: ./g_channel_demo/QAM16_SNR19.png
已保存: ./g_channel_demo/QAM16_SNR21.png
已保存: ./g_channel_demo/QAM16_SNR23.png
已保存: ./g_channel_demo/QAM16_SNR25.png
已保存: ./g_channel_demo/QAM16_SNR27.png
已保存: ./g_channel_demo/QAM16_SNR29.png
已保存: ./g_channel_demo/QAM16_SNR31.png
已保存: ./g_channel_demo/QAM16_SNR33.png
已保存: ./g_channel_demo/QAM16_SNR35.png
已保存: ./g_channel_demo/QAM16_SNR37.png
已保存: ./g_channel_demo/QAM16_SNR39.png
已保存: ./g_channel_demo/QAM16_SNR41.png
已保存: ./g_channel_demo/QAM64_SNR-5.png
已保存: ./g_channel_demo/QAM64_SNR-3.png
已保存: ./g_channel_demo/QAM64_SNR-1.png
已保存: ./g_channel_demo/QAM64_SNR1.png
已保存: ./g_channel_d

C:\Users\CPC-ML\AppData\Local\Temp\ipykernel_13956\1307707760.py:63: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  syms = ((idx // 8) * 2 - 7) + 1j*((idx % 8) * 2 - 7)


已保存: ./g_channel_demo/QAM64_SNR7.png
已保存: ./g_channel_demo/QAM64_SNR9.png
已保存: ./g_channel_demo/QAM64_SNR11.png
已保存: ./g_channel_demo/QAM64_SNR13.png
已保存: ./g_channel_demo/QAM64_SNR15.png
已保存: ./g_channel_demo/QAM64_SNR17.png
已保存: ./g_channel_demo/QAM64_SNR19.png
已保存: ./g_channel_demo/QAM64_SNR21.png
已保存: ./g_channel_demo/QAM64_SNR23.png
已保存: ./g_channel_demo/QAM64_SNR25.png
已保存: ./g_channel_demo/QAM64_SNR27.png
已保存: ./g_channel_demo/QAM64_SNR29.png
已保存: ./g_channel_demo/QAM64_SNR31.png
已保存: ./g_channel_demo/QAM64_SNR33.png
已保存: ./g_channel_demo/QAM64_SNR35.png
已保存: ./g_channel_demo/QAM64_SNR37.png
已保存: ./g_channel_demo/QAM64_SNR39.png
已保存: ./g_channel_demo/QAM64_SNR41.png
已保存: ./g_channel_demo/FSK2_SNR-5.png
已保存: ./g_channel_demo/FSK2_SNR-3.png
已保存: ./g_channel_demo/FSK2_SNR-1.png
已保存: ./g_channel_demo/FSK2_SNR1.png
已保存: ./g_channel_demo/FSK2_SNR3.png
已保存: ./g_channel_demo/FSK2_SNR5.png
已保存: ./g_channel_demo/FSK2_SNR7.png
已保存: ./g_channel_demo/FSK2_SNR9.png
已保存: ./g_channel_demo/FSK2_

B 通道 的核心逻辑与 G 通道完全一致，但它关注的是信号的虚部 (Imaginary Part / Quadrature Component)。（虚部动力学轨迹）

*   **物理定义**：它绘制的是信号虚部在当前时刻的值 $Q[t]$ 与前一时刻的值 $Q[t-1]$ 之间的关系。
    *   X 轴：$Q[t]$
    *   Y 轴：$Q[t-1]$
*   **物理意义**：
    *   **对称性互补**：复数信号包含实部和虚部。对于 BPSK 这样的信号，如果相位没有旋转（0度），信息全在实部，虚部只有噪声；但如果信号旋转了 90 度，信息就全跑到了虚部。
    *   **防旋转机制**：G 通道和 B 通道互为**“备份”**。无论信号的相位如何旋转，信号的动力学特征（是突变还是渐变）总会保留在 G 通道或 B 通道中的某一个（或者两者都有）。
    *   **正交投影**：相当于从侧面看这个信号的“运动轨迹”。

#### 2. 数学推导

输入信号序列为 $s[n] = I[n] + jQ[n]$。对于 B 通道，我们只提取虚部 $Q[n]$。

**步骤 1：构造延迟向量**
构建点对 $(x, y)$：
$$ x[n] = Q[n] $$
$$ y[n] = Q[n-1] $$
(代码中取 `imag[1:]` 和 `imag[:-1]`)

**步骤 2：离散化与映射**
设定观察范围 $[-B, B]$，映射到 $512 \times 512$ 的网格：
$$ u[n] = \text{Map}(Q[n]) $$
$$ v[n] = \text{Map}(Q[n-1]) $$

**步骤 3：密度统计 (2D Histogram)**
$$ \mathbf{B}_{u,v} = \sum_{n=1}^{N-1} \delta(u[n] - u, v[n] - v) $$

**步骤 4：对数增强**
$$ \mathbf{B}_{final} = \text{Normalize}(\log(1 + \mathbf{B})) $$



#### 3. 图像结果分析 (预期形状)

*   **BPSK**：
    *   **情形 A (无旋转)**：实部在跳变，虚部几乎为 0（只有噪声）。B 通道会显示为一个**中心圆团**（噪声球）。
    *   **情形 B (有旋转)**：如果信号旋转了，虚部开始有分量。B 通道会逐渐显现出**哑铃状**或**双斑点**结构。
*   **FSK**：
    *   虚部 $Q(t) = \sin(\omega t + \phi)$。
    *   正弦波的延迟嵌入（$sin(t)$ vs $sin(t-1)$）依然是一个**椭圆**。
    *   **结论**：无论 G 通道还是 B 通道，FSK 永远是空心的环/椭圆。这是最稳定的特征。
*   **QAM**：
    *   类似于 G 通道，呈现复杂的网格状跳变轨迹。

In [11]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg') # 静默模式
import matplotlib.pyplot as plt
import os

# ================= 配置 =================
CLASS_NAMES = ["BPSK", "QPSK", "PSK8", "QAM16", "QAM64", "FSK2"]
# SNR_LEVELS = [-5, 2, 10, 18, 25]
SNR_LEVELS = [-5, -3, -1, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41]   
IMG_SIZE = 512
SAMPLES = 10240

SAVE_DIR = "./b_channel_demo"
os.makedirs(SAVE_DIR, exist_ok=True)

# ================= 信号生成 (保持高保真逻辑) =================
def generate_rrc_filter(N_taps, beta, sps):
    t = torch.arange(-(N_taps - 1) // 2, (N_taps - 1) // 2 + 1, dtype=torch.float32) / sps
    mid_idx = (N_taps - 1) // 2
    t[mid_idx] = 1e-8
    num = torch.sin(np.pi * t * (1 - beta)) + 4 * beta * t * torch.cos(np.pi * t * (1 + beta))
    den = np.pi * t * (1 - (4 * beta * t)**2)
    coeffs = num / den
    coeffs[mid_idx] = 1 - beta + (4 * beta / np.pi)
    return coeffs / torch.sqrt(torch.sum(coeffs**2))

def add_noise_and_imperfect(sig, snr_db):
    N = len(sig)
    # 保持微小频偏和随机相位
    freq_offset = np.random.uniform(-50, 50) 
    t = torch.arange(N, dtype=torch.float32) / 80e3
    phase_offset = np.random.uniform(0, 2*np.pi)
    rotation = torch.exp(1j * (2 * np.pi * freq_offset * t + phase_offset))
    sig = sig * rotation
    
    sig_power = torch.mean(torch.abs(sig)**2)
    noise_power = sig_power / (10**(snr_db/10.0))
    noise = (torch.randn_like(sig) + 1j*torch.randn_like(sig)) * torch.sqrt(noise_power/2)
    return sig + noise

def generate_signal_for_demo(mod_type, snr_db):
    N = SAMPLES
    Fs, Rs = 80e3, 20e3
    sps = int(Fs / Rs)
    num_symbols = int(N / sps) + 50
    
    if mod_type == "BPSK":
        syms = 2*torch.randint(0, 2, (num_symbols,)) - 1 + 0j
    elif mod_type == "QPSK":
        phase = np.pi/4 + torch.randint(0, 4, (num_symbols,)) * np.pi/2
        syms = torch.exp(1j * phase)
    elif mod_type == "PSK8":
        phase = torch.randint(0, 8, (num_symbols,)) * 2*np.pi/8
        syms = torch.exp(1j * phase)
    elif mod_type == "QAM16":
        idx = torch.randint(0, 16, (num_symbols,))
        syms = ((idx // 4) * 2 - 3) + 1j*((idx % 4) * 2 - 3)
    elif mod_type == "QAM64":
        idx = torch.randint(0, 64, (num_symbols,))
        syms = ((idx // 8) * 2 - 7) + 1j*((idx % 8) * 2 - 7)
    elif mod_type == "FSK2":
        bits = torch.randint(0, 2, (N + 100,))
        f_dev = np.random.uniform(500, 6000)
        freqs = (bits * 2 - 1) * f_dev
        phase = 2 * np.pi * torch.cumsum(freqs/Fs, dim=0)
        sig = torch.exp(1j * phase)[:N]
        return add_noise_and_imperfect(sig, snr_db)

    upsampled = torch.zeros(num_symbols * sps, dtype=torch.complex64)
    upsampled[::sps] = syms
    taps = 10 * sps + 1
    h = generate_rrc_filter(taps, 0.35, sps)
    real = F.conv1d(upsampled.real.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    imag = F.conv1d(upsampled.imag.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    sig = (real + 1j*imag).squeeze()
    if len(sig) > N: sig = sig[:N]
    
    return add_noise_and_imperfect(sig, snr_db)

# ================= B 通道计算核心 (Imag Trajectory) =================
def compute_b_channel(sig):
    """
    计算 B 通道：虚部延迟嵌入 (Q[t] vs Q[t-1])
    Input: Complex Tensor (N,)
    Output: 2D Tensor (512, 512)
    """
    # 1. 归一化 (使用复数模长统一归一化，保持 I/Q 比例)
    scale = torch.sqrt(torch.mean(sig.real**2 + sig.imag**2)) + 1e-9
    norm_sig = sig / (3.0 * scale)
    
    # 只取虚部
    imag = norm_sig.imag
    
    # 2. 构造延迟对
    x = imag[1:]
    y = imag[:-1]
    
    range_min, range_max = -1.2, 1.2
    x = torch.clamp(x, range_min, range_max)
    y = torch.clamp(y, range_min, range_max)
    
    # 3. 映射到 grid
    bins = IMG_SIZE
    x_idx = ((x - range_min) / (range_max - range_min) * (bins - 1)).long()
    y_idx = ((y - range_min) / (range_max - range_min) * (bins - 1)).long()
    
    # 4. 统计频次
    flat_idx = y_idx * bins + x_idx
    hist = torch.bincount(flat_idx, minlength=bins*bins).float()
    hist = hist.view(bins, bins)
    
    # 5. 对数压缩 + 归一化
    hist = torch.log1p(hist)
    hist = (hist - hist.min()) / (hist.max() - hist.min() + 1e-8)
    
    return torch.flip(hist, [0])

# ================= 主流程 =================
def main():
    print(f"开始生成 B 通道 (虚部相空间) 演示图...")
    print(f"分辨率: {IMG_SIZE}x{IMG_SIZE}, 路径: {os.path.abspath(SAVE_DIR)}")
    
    fig, axes = plt.subplots(len(CLASS_NAMES), len(SNR_LEVELS), figsize=(20, 24))
    
    for r, mod in enumerate(CLASS_NAMES):
        for c, snr in enumerate(SNR_LEVELS):
            sig = generate_signal_for_demo(mod, snr)
            
            # 计算 B 通道
            heatmap = compute_b_channel(sig)
            
            # 保存单张
            filename = f"{SAVE_DIR}/{mod}_SNR{snr}.png"
            plt.imsave(filename, heatmap.numpy(), cmap='gray')
            print(f"已保存: {filename}")
            
            # 总览
            ax = axes[r, c]
            ax.imshow(heatmap.numpy(), cmap='inferno')
            if r == 0: ax.set_title(f"SNR = {snr} dB", fontsize=16)
            if c == 0: ax.set_ylabel(mod, fontsize=20, fontweight='bold')
            ax.set_xticks([])
            ax.set_yticks([])
            
    # 保存总览
    plt.savefig(f"{SAVE_DIR}/B_Channel_Overview.png", dpi=100, bbox_inches='tight')
    plt.close(fig)
    print("全部完成！")

if __name__ == "__main__":
    main()

开始生成 B 通道 (虚部相空间) 演示图...
分辨率: 512x512, 路径: e:\Project\AMR-Benchmark\Other\b_channel_demo
已保存: ./b_channel_demo/BPSK_SNR-5.png
已保存: ./b_channel_demo/BPSK_SNR-3.png
已保存: ./b_channel_demo/BPSK_SNR-1.png
已保存: ./b_channel_demo/BPSK_SNR1.png
已保存: ./b_channel_demo/BPSK_SNR3.png
已保存: ./b_channel_demo/BPSK_SNR5.png
已保存: ./b_channel_demo/BPSK_SNR7.png
已保存: ./b_channel_demo/BPSK_SNR9.png
已保存: ./b_channel_demo/BPSK_SNR11.png
已保存: ./b_channel_demo/BPSK_SNR13.png
已保存: ./b_channel_demo/BPSK_SNR15.png
已保存: ./b_channel_demo/BPSK_SNR17.png
已保存: ./b_channel_demo/BPSK_SNR19.png
已保存: ./b_channel_demo/BPSK_SNR21.png
已保存: ./b_channel_demo/BPSK_SNR23.png
已保存: ./b_channel_demo/BPSK_SNR25.png
已保存: ./b_channel_demo/BPSK_SNR27.png
已保存: ./b_channel_demo/BPSK_SNR29.png
已保存: ./b_channel_demo/BPSK_SNR31.png
已保存: ./b_channel_demo/BPSK_SNR33.png
已保存: ./b_channel_demo/BPSK_SNR35.png
已保存: ./b_channel_demo/BPSK_SNR37.png
已保存: ./b_channel_demo/BPSK_SNR39.png
已保存: ./b_channel_demo/BPSK_SNR41.png
已保存: ./b_channel_demo/QPSK_S

C:\Users\CPC-ML\AppData\Local\Temp\ipykernel_13956\1109778737.py:60: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  syms = ((idx // 4) * 2 - 3) + 1j*((idx % 4) * 2 - 3)


已保存: ./b_channel_demo/QAM16_SNR1.png
已保存: ./b_channel_demo/QAM16_SNR3.png
已保存: ./b_channel_demo/QAM16_SNR5.png
已保存: ./b_channel_demo/QAM16_SNR7.png
已保存: ./b_channel_demo/QAM16_SNR9.png
已保存: ./b_channel_demo/QAM16_SNR11.png
已保存: ./b_channel_demo/QAM16_SNR13.png
已保存: ./b_channel_demo/QAM16_SNR15.png
已保存: ./b_channel_demo/QAM16_SNR17.png
已保存: ./b_channel_demo/QAM16_SNR19.png
已保存: ./b_channel_demo/QAM16_SNR21.png
已保存: ./b_channel_demo/QAM16_SNR23.png
已保存: ./b_channel_demo/QAM16_SNR25.png
已保存: ./b_channel_demo/QAM16_SNR27.png
已保存: ./b_channel_demo/QAM16_SNR29.png
已保存: ./b_channel_demo/QAM16_SNR31.png
已保存: ./b_channel_demo/QAM16_SNR33.png
已保存: ./b_channel_demo/QAM16_SNR35.png
已保存: ./b_channel_demo/QAM16_SNR37.png
已保存: ./b_channel_demo/QAM16_SNR39.png
已保存: ./b_channel_demo/QAM16_SNR41.png
已保存: ./b_channel_demo/QAM64_SNR-5.png
已保存: ./b_channel_demo/QAM64_SNR-3.png
已保存: ./b_channel_demo/QAM64_SNR-1.png
已保存: ./b_channel_demo/QAM64_SNR1.png
已保存: ./b_channel_demo/QAM64_SNR3.png
已保存: ./b_channel_de

C:\Users\CPC-ML\AppData\Local\Temp\ipykernel_13956\1109778737.py:63: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  syms = ((idx // 8) * 2 - 7) + 1j*((idx % 8) * 2 - 7)


已保存: ./b_channel_demo/QAM64_SNR11.png
已保存: ./b_channel_demo/QAM64_SNR13.png
已保存: ./b_channel_demo/QAM64_SNR15.png
已保存: ./b_channel_demo/QAM64_SNR17.png
已保存: ./b_channel_demo/QAM64_SNR19.png
已保存: ./b_channel_demo/QAM64_SNR21.png
已保存: ./b_channel_demo/QAM64_SNR23.png
已保存: ./b_channel_demo/QAM64_SNR25.png
已保存: ./b_channel_demo/QAM64_SNR27.png
已保存: ./b_channel_demo/QAM64_SNR29.png
已保存: ./b_channel_demo/QAM64_SNR31.png
已保存: ./b_channel_demo/QAM64_SNR33.png
已保存: ./b_channel_demo/QAM64_SNR35.png
已保存: ./b_channel_demo/QAM64_SNR37.png
已保存: ./b_channel_demo/QAM64_SNR39.png
已保存: ./b_channel_demo/QAM64_SNR41.png
已保存: ./b_channel_demo/FSK2_SNR-5.png
已保存: ./b_channel_demo/FSK2_SNR-3.png
已保存: ./b_channel_demo/FSK2_SNR-1.png
已保存: ./b_channel_demo/FSK2_SNR1.png
已保存: ./b_channel_demo/FSK2_SNR3.png
已保存: ./b_channel_demo/FSK2_SNR5.png
已保存: ./b_channel_demo/FSK2_SNR7.png
已保存: ./b_channel_demo/FSK2_SNR9.png
已保存: ./b_channel_demo/FSK2_SNR11.png
已保存: ./b_channel_demo/FSK2_SNR13.png
已保存: ./b_channel_demo/FSK2_

1. A 通道：应对R/G/B 通道在面对严重的载波频偏（CFO）时可能会发生旋转、变形
极坐标分布


**A 通道** 的核心是将信号从笛卡尔坐标系（I/Q）转换到**极坐标系（Polar Coordinates）**。

*   **物理定义**：它绘制的是信号的**相位（Phase）**与**幅度（Magnitude）**的联合分布。
    *   **X 轴**：瞬时相位 $\theta(t)$，范围 $[-\pi, \pi]$。
    *   **Y 轴**：瞬时幅度 $A(t) = \sqrt{I^2 + Q^2}$。
*   **物理意义**：
    *   **幅度分层（Amplitude Slicing）**：这是区分 **PSK** 和 **QAM** 的绝对标准。
        *   **PSK (BPSK/QPSK/8PSK)**：所有符号都在同一个圆上，**幅度恒定**。在 A 通道上，它们位于同一水平高度。
        *   **QAM (16/64)**：符号分布在不同的圆上，**幅度多级分层**。在 A 通道上，你会看到几层不同的水平条带。
    *   **抗旋转特性**：当信号存在频偏时，星座图（R通道）会旋转。但在极坐标下，旋转只是相位 $\theta$ 在增加，而**幅度 $A$ 是不变的**。
        *   这意味着：R 通道里的点转成了圈，但在 A 通道里，**它依然是一条水平线**。


输入信号序列为 $s[n] = I[n] + jQ[n]$。

**步骤 1：坐标转换**
$$ r[n] = |s[n]| = \sqrt{I[n]^2 + Q[n]^2} $$
$$ \phi[n] = \text{angle}(s[n]) \in [-\pi, \pi] $$

**步骤 2：归一化**
*   **相位**：归一化到 $[-1, 1]$
    $$ u[n] = \phi[n] / \pi $$
*   **幅度**：使用统计归一化（3倍均方根），并限制关注范围（例如 $[0, 1.5]$）
    $$ v[n] = r[n] / (3 \times \text{RMS}) $$

**步骤 3：密度统计 (2D Histogram)**
$$ \mathbf{A}_{u,v} = \sum_{n=0}^{N-1} \delta(u[n] - u, v[n] - v) $$

**步骤 4：对数增强**
$$ \mathbf{A}_{final} = \text{Normalize}(\log(1 + \mathbf{A})) $$


*   **BPSK**：
    *   **形状**：**两根垂直的柱子**（位于相位 $0$ 和 $\pi$ 附近，高度恒定）。
    *   **如果有频偏**：柱子会变宽，或者沿 X 轴平移，但依然是**单层**的。
*   **QPSK**：
    *   **形状**：**四根垂直的柱子**。
*   **QAM16**：
    *   **形状**：**三层斑点**。因为 QAM16 的星座点分布在 3 个不同的幅度圆环上。这是识别 QAM 最强的证据。
*   **FSK**：
    *   **形状**：**一条贯穿整个 X 轴的水平线**。
    *   **原因**：FSK 是恒定包络调制（幅度不变），但频率在变，意味着相位 $\phi(t)$ 在 $0$ 到 $2\pi$ 之间快速且连续地扫描。所以它铺满了整个相位轴（X轴），但固定在某个幅度轴（Y轴）。

In [12]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg') # 静默模式
import matplotlib.pyplot as plt
import os

# ================= 配置 =================
CLASS_NAMES = ["BPSK", "QPSK", "PSK8", "QAM16", "QAM64", "FSK2"]
# SNR_LEVELS = [-5, 2, 10, 18, 25]
SNR_LEVELS = [-5, -3, -1, 1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41]   
IMG_SIZE = 512
SAMPLES = 10240

SAVE_DIR = "./a_channel_demo"
os.makedirs(SAVE_DIR, exist_ok=True)

# ================= 信号生成 (复用高保真逻辑) =================
def generate_rrc_filter(N_taps, beta, sps):
    t = torch.arange(-(N_taps - 1) // 2, (N_taps - 1) // 2 + 1, dtype=torch.float32) / sps
    mid_idx = (N_taps - 1) // 2
    t[mid_idx] = 1e-8
    num = torch.sin(np.pi * t * (1 - beta)) + 4 * beta * t * torch.cos(np.pi * t * (1 + beta))
    den = np.pi * t * (1 - (4 * beta * t)**2)
    coeffs = num / den
    coeffs[mid_idx] = 1 - beta + (4 * beta / np.pi)
    return coeffs / torch.sqrt(torch.sum(coeffs**2))

def add_noise_and_imperfect(sig, snr_db):
    N = len(sig)
    # 微频偏 (这对 A 通道影响很有趣，会观察到柱子平移)
    freq_offset = np.random.uniform(-50, 50) 
    t = torch.arange(N, dtype=torch.float32) / 80e3
    phase_offset = np.random.uniform(0, 2*np.pi)
    rotation = torch.exp(1j * (2 * np.pi * freq_offset * t + phase_offset))
    sig = sig * rotation
    
    sig_power = torch.mean(torch.abs(sig)**2)
    noise_power = sig_power / (10**(snr_db/10.0))
    noise = (torch.randn_like(sig) + 1j*torch.randn_like(sig)) * torch.sqrt(noise_power/2)
    return sig + noise

def generate_signal_for_demo(mod_type, snr_db):
    N = SAMPLES
    Fs, Rs = 80e3, 20e3
    sps = int(Fs / Rs)
    num_symbols = int(N / sps) + 50
    
    if mod_type == "BPSK":
        syms = 2*torch.randint(0, 2, (num_symbols,)) - 1 + 0j
    elif mod_type == "QPSK":
        phase = np.pi/4 + torch.randint(0, 4, (num_symbols,)) * np.pi/2
        syms = torch.exp(1j * phase)
    elif mod_type == "PSK8":
        phase = torch.randint(0, 8, (num_symbols,)) * 2*np.pi/8
        syms = torch.exp(1j * phase)
    elif mod_type == "QAM16":
        idx = torch.randint(0, 16, (num_symbols,))
        syms = ((idx // 4) * 2 - 3) + 1j*((idx % 4) * 2 - 3)
    elif mod_type == "QAM64":
        idx = torch.randint(0, 64, (num_symbols,))
        syms = ((idx // 8) * 2 - 7) + 1j*((idx % 8) * 2 - 7)
    elif mod_type == "FSK2":
        bits = torch.randint(0, 2, (N + 100,))
        f_dev = np.random.uniform(500, 6000)
        freqs = (bits * 2 - 1) * f_dev
        phase = 2 * np.pi * torch.cumsum(freqs/Fs, dim=0)
        sig = torch.exp(1j * phase)[:N]
        return add_noise_and_imperfect(sig, snr_db)

    upsampled = torch.zeros(num_symbols * sps, dtype=torch.complex64)
    upsampled[::sps] = syms
    taps = 10 * sps + 1
    h = generate_rrc_filter(taps, 0.35, sps)
    real = F.conv1d(upsampled.real.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    imag = F.conv1d(upsampled.imag.view(1,1,-1), h.view(1,1,-1), padding=taps//2)
    sig = (real + 1j*imag).squeeze()
    if len(sig) > N: sig = sig[:N]
    
    return add_noise_and_imperfect(sig, snr_db)

# ================= A 通道计算核心 (Polar Density) =================
def compute_a_channel(sig):
    """
    计算 A 通道：极坐标密度 (Phase vs Magnitude)
    X轴: 相位 [-pi, pi]
    Y轴: 幅度 [0, 1.5]
    Output: 2D Tensor (512, 512)
    """
    # 1. 归一化
    scale = torch.sqrt(torch.mean(sig.real**2 + sig.imag**2)) + 1e-9
    norm_sig = sig / (3.0 * scale)
    
    # 2. 提取极坐标分量
    # 相位: [-1, 1]
    phase = torch.angle(norm_sig) / np.pi 
    # 幅度: [0, inf)
    mag = torch.abs(norm_sig)
    
    # 3. 映射到 grid
    # X轴范围: [-1.0, 1.0] (完整相位周期)
    # Y轴范围: [0.0, 1.5] (归一化后的幅度，通常在 0.3~1.2 之间)
    x = phase
    y = mag
    
    x_range = (-1.0, 1.0)
    y_range = (0.0, 1.5)
    
    x = torch.clamp(x, x_range[0], x_range[1])
    y = torch.clamp(y, y_range[0], y_range[1])
    
    bins = IMG_SIZE
    x_idx = ((x - x_range[0]) / (x_range[1] - x_range[0]) * (bins - 1)).long()
    y_idx = ((y - y_range[0]) / (y_range[1] - y_range[0]) * (bins - 1)).long()
    
    # 4. 统计频次
    flat_idx = y_idx * bins + x_idx
    hist = torch.bincount(flat_idx, minlength=bins*bins).float()
    hist = hist.view(bins, bins)
    
    # 5. 对数压缩 + 归一化
    hist = torch.log1p(hist)
    hist = (hist - hist.min()) / (hist.max() - hist.min() + 1e-8)
    
    return torch.flip(hist, [0]) # Y轴向上 (幅度向上增加)

# ================= 主流程 =================
def main():
    print(f"开始生成 A 通道 (极坐标分布) 演示图...")
    print(f"分辨率: {IMG_SIZE}x{IMG_SIZE}, 路径: {os.path.abspath(SAVE_DIR)}")
    
    fig, axes = plt.subplots(len(CLASS_NAMES), len(SNR_LEVELS), figsize=(20, 24))
    
    for r, mod in enumerate(CLASS_NAMES):
        for c, snr in enumerate(SNR_LEVELS):
            sig = generate_signal_for_demo(mod, snr)
            
            # 计算 A 通道
            heatmap = compute_a_channel(sig)
            
            # 保存单张
            filename = f"{SAVE_DIR}/{mod}_SNR{snr}.png"
            plt.imsave(filename, heatmap.numpy(), cmap='gray')
            print(f"已保存: {filename}")
            
            # 总览
            ax = axes[r, c]
            ax.imshow(heatmap.numpy(), cmap='inferno')
            if r == 0: ax.set_title(f"SNR = {snr} dB", fontsize=16)
            if c == 0: ax.set_ylabel(mod, fontsize=20, fontweight='bold')
            ax.set_xticks([])
            ax.set_yticks([])
            
    # 保存总览
    plt.savefig(f"{SAVE_DIR}/A_Channel_Overview.png", dpi=100, bbox_inches='tight')
    plt.close(fig)
    print("全部完成！")

if __name__ == "__main__":
    main()

开始生成 A 通道 (极坐标分布) 演示图...
分辨率: 512x512, 路径: e:\Project\AMR-Benchmark\Other\a_channel_demo
已保存: ./a_channel_demo/BPSK_SNR-5.png
已保存: ./a_channel_demo/BPSK_SNR-3.png
已保存: ./a_channel_demo/BPSK_SNR-1.png
已保存: ./a_channel_demo/BPSK_SNR1.png
已保存: ./a_channel_demo/BPSK_SNR3.png
已保存: ./a_channel_demo/BPSK_SNR5.png
已保存: ./a_channel_demo/BPSK_SNR7.png
已保存: ./a_channel_demo/BPSK_SNR9.png
已保存: ./a_channel_demo/BPSK_SNR11.png
已保存: ./a_channel_demo/BPSK_SNR13.png
已保存: ./a_channel_demo/BPSK_SNR15.png
已保存: ./a_channel_demo/BPSK_SNR17.png
已保存: ./a_channel_demo/BPSK_SNR19.png
已保存: ./a_channel_demo/BPSK_SNR21.png
已保存: ./a_channel_demo/BPSK_SNR23.png
已保存: ./a_channel_demo/BPSK_SNR25.png
已保存: ./a_channel_demo/BPSK_SNR27.png
已保存: ./a_channel_demo/BPSK_SNR29.png
已保存: ./a_channel_demo/BPSK_SNR31.png
已保存: ./a_channel_demo/BPSK_SNR33.png
已保存: ./a_channel_demo/BPSK_SNR35.png
已保存: ./a_channel_demo/BPSK_SNR37.png
已保存: ./a_channel_demo/BPSK_SNR39.png
已保存: ./a_channel_demo/BPSK_SNR41.png
已保存: ./a_channel_demo/QPSK_S

C:\Users\CPC-ML\AppData\Local\Temp\ipykernel_13956\775894674.py:60: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  syms = ((idx // 4) * 2 - 3) + 1j*((idx % 4) * 2 - 3)


已保存: ./a_channel_demo/QAM16_SNR-5.png
已保存: ./a_channel_demo/QAM16_SNR-3.png
已保存: ./a_channel_demo/QAM16_SNR-1.png
已保存: ./a_channel_demo/QAM16_SNR1.png
已保存: ./a_channel_demo/QAM16_SNR3.png
已保存: ./a_channel_demo/QAM16_SNR5.png
已保存: ./a_channel_demo/QAM16_SNR7.png
已保存: ./a_channel_demo/QAM16_SNR9.png
已保存: ./a_channel_demo/QAM16_SNR11.png
已保存: ./a_channel_demo/QAM16_SNR13.png
已保存: ./a_channel_demo/QAM16_SNR15.png
已保存: ./a_channel_demo/QAM16_SNR17.png
已保存: ./a_channel_demo/QAM16_SNR19.png
已保存: ./a_channel_demo/QAM16_SNR21.png
已保存: ./a_channel_demo/QAM16_SNR23.png
已保存: ./a_channel_demo/QAM16_SNR25.png
已保存: ./a_channel_demo/QAM16_SNR27.png
已保存: ./a_channel_demo/QAM16_SNR29.png
已保存: ./a_channel_demo/QAM16_SNR31.png
已保存: ./a_channel_demo/QAM16_SNR33.png
已保存: ./a_channel_demo/QAM16_SNR35.png
已保存: ./a_channel_demo/QAM16_SNR37.png
已保存: ./a_channel_demo/QAM16_SNR39.png
已保存: ./a_channel_demo/QAM16_SNR41.png
已保存: ./a_channel_demo/QAM64_SNR-5.png
已保存: ./a_channel_demo/QAM64_SNR-3.png
已保存: ./a_channel_

C:\Users\CPC-ML\AppData\Local\Temp\ipykernel_13956\775894674.py:63: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  syms = ((idx // 8) * 2 - 7) + 1j*((idx % 8) * 2 - 7)


已保存: ./a_channel_demo/QAM64_SNR1.png
已保存: ./a_channel_demo/QAM64_SNR3.png
已保存: ./a_channel_demo/QAM64_SNR5.png
已保存: ./a_channel_demo/QAM64_SNR7.png
已保存: ./a_channel_demo/QAM64_SNR9.png
已保存: ./a_channel_demo/QAM64_SNR11.png
已保存: ./a_channel_demo/QAM64_SNR13.png
已保存: ./a_channel_demo/QAM64_SNR15.png
已保存: ./a_channel_demo/QAM64_SNR17.png
已保存: ./a_channel_demo/QAM64_SNR19.png
已保存: ./a_channel_demo/QAM64_SNR21.png
已保存: ./a_channel_demo/QAM64_SNR23.png
已保存: ./a_channel_demo/QAM64_SNR25.png
已保存: ./a_channel_demo/QAM64_SNR27.png
已保存: ./a_channel_demo/QAM64_SNR29.png
已保存: ./a_channel_demo/QAM64_SNR31.png
已保存: ./a_channel_demo/QAM64_SNR33.png
已保存: ./a_channel_demo/QAM64_SNR35.png
已保存: ./a_channel_demo/QAM64_SNR37.png
已保存: ./a_channel_demo/QAM64_SNR39.png
已保存: ./a_channel_demo/QAM64_SNR41.png
已保存: ./a_channel_demo/FSK2_SNR-5.png
已保存: ./a_channel_demo/FSK2_SNR-3.png
已保存: ./a_channel_demo/FSK2_SNR-1.png
已保存: ./a_channel_demo/FSK2_SNR1.png
已保存: ./a_channel_demo/FSK2_SNR3.png
已保存: ./a_channel_demo/FS